# Logistic Regression Assumptions — Tourism Solution

**Short name (GitHub):** `LogReg_Tour`  
Work the skeleton first. Numbers: scikit-learn 1.x, `penalty=None`, `random_state=0` (model) and `random_state=6` (imbalance). Hospitality cancel models are noisy — expect AUC in the high 0.60s, not 0.98.


## Inline cheat-sheet

| Item | This file |
|------|-----------|
| Rows / positivity | 740 bookings, CANCEL=273, STAY=467, rate ≈ 0.369 |
| Independence | unique `booking_id` — True |
| 10-EPV | 273/10 = 27.3 |
| Outlier cut | `special_requests` 99th pct ≈ 8.7, 8 rows dropped |
| Collinear pair | `nights` ~ `booking_value` (r ≈ 0.76); value ≈ nights × ADR (r ≈ 0.996) |
| Weak logit feature | `checkin_hour` |
| Core 5-feature test (rs=0) | acc≈0.65 · prec≈0.52 · rec≈0.36 · F1≈0.43 · AUC≈0.68 |
| CM @0.50 | TN 115, FP 27, FN 51, TP 29 |
| CM @0.25 | FN 8, FP 102 |
| CM @0.75 | FN 74, FP 1 |
| Balanced weights (rs=6) | recall 0.37 → **0.68** |


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score,
)
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

%matplotlib inline
sns.set_style("whitegrid")
np.set_printoptions(precision=4, suppress=True)
print("libraries ready")


## 1. Load and encode

In [ ]:
df = pd.read_csv("data/tour_hotels.csv")
df["cancelled"] = df["cancelled"].map({"CANCEL": 1, "STAY": 0}).astype(int)
print(df.head())
print(df.cancelled.value_counts())


## 2. Assumptions I

In [ ]:
print(df.cancelled.value_counts())
print("n classes:", df.cancelled.nunique())
unique_ids = df.booking_id.nunique() == df.booking_id.count()
print(unique_ids)
max_features = min(df.cancelled.value_counts()) / 10
print(max_features)  # 27.3


In [ ]:
all_features = [
    "lead_time", "nights", "adr", "booking_value", "party_size", "prev_cancels",
    "special_requests", "is_weekend", "online_channel", "has_deposit", "checkin_hour",
]
cont_features = [
    "lead_time", "nights", "adr", "booking_value", "party_size",
    "prev_cancels", "special_requests", "checkin_hour",
]
plt.figure(figsize=(10, 4.2))
sns.boxplot(data=np.log(df[cont_features] + 0.01).apply(zscore))
plt.xticks(rotation=40, ha="right")
plt.title("Log-z boxplot — special_requests has the long upper tail")
plt.tight_layout(); plt.show()


In [ ]:
q_hi = df["special_requests"].quantile(0.99)
df_filtered = df[df["special_requests"] < q_hi].copy()
print("q99 =", q_hi, " kept =", len(df_filtered), " dropped =", len(df) - len(df_filtered))
plt.figure(figsize=(10, 4.2))
sns.boxplot(data=np.log(df_filtered[cont_features] + 0.01).apply(zscore))
plt.xticks(rotation=40, ha="right")
plt.title("After 99th-percentile filter on special_requests")
plt.tight_layout(); plt.show()


## 3. Assumptions II

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.regplot(x="lead_time", y="cancelled", data=df, logistic=True, ax=axes[0],
            scatter_kws={"alpha": 0.22, "s": 14})
axes[0].set_title("lead_time — sigmoid-like")
sns.regplot(x="checkin_hour", y="cancelled", data=df, logistic=True, ax=axes[1],
            scatter_kws={"alpha": 0.22, "s": 14})
axes[1].set_title("checkin_hour — flat / weak")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[all_features].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, annot_kws={"size": 7})
plt.title("Feature correlations")
plt.tight_layout(); plt.show()
correlated_pair = ["nights", "booking_value"]  # value ≈ nights × ADR
print("correlated_pair:", correlated_pair)


## 4. scikit-learn

In [ ]:
core = ["lead_time", "prev_cancels", "online_channel", "has_deposit", "special_requests"]
outcome = "cancelled"
x_train, x_test, y_train, y_test = train_test_split(
    df[core], df[outcome], random_state=0, test_size=0.3
)
log_reg = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000, solver="lbfgs")
print(log_reg.get_params())
log_reg.fit(x_train, y_train)
coefficients = log_reg.coef_
intercept = log_reg.intercept_
print("coefficients:", coefficients)
print("intercept:", intercept)
# lead_time +, prev_cancels +, online_channel +, has_deposit −, special_requests −


In [ ]:
y_pred = log_reg.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"accuracy\t{accuracy:.4f}")
print(f"precision\t{precision:.4f}")
print(f"recall   \t{recall:.4f}")
print(f"f1       \t{f1:.4f}")
# acc ≈ 0.649  prec ≈ 0.518  rec ≈ 0.362  f1 ≈ 0.426  — noisy, as hospitality cancels are


In [ ]:
test_conf_matrix = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual stay", "actual cancel"],
    columns=["predicted stay", "predicted cancel"],
)
print(test_conf_matrix)
# TN 115, FP 27, FN 51, TP 29


## 5. Thresholds

In [ ]:
y_pred_prob = log_reg.predict_proba(x_test)
y_pred_class = (y_pred_prob[:, 1] > 0.5) * 1.0
print("same as predict()?", np.array_equal(y_pred_class, y_pred))
print("CM 50%"); print(confusion_matrix(y_test, y_pred_class))
print("CM 25%"); print(confusion_matrix(y_test, (y_pred_prob[:, 1] > 0.25) * 1.0))
print("CM 75%"); print(confusion_matrix(y_test, (y_pred_prob[:, 1] > 0.75) * 1.0))
# 25%: FN 8  FP 102 — hold more inventory
# 50%: FN 51 FP 27
# 75%: FN 74 FP 1  — almost never flags a cancel


In [ ]:
thresh = np.linspace(0, 1, 100)
false_negatives = []
for t in thresh:
    cm = confusion_matrix(y_test, (y_pred_prob[:, 1] > t) * 1.0)
    false_negatives.append(cm[1, 0])
thresh_choice = thresh[np.argmax(np.array(false_negatives) >= 12)]
print("thresh_choice =", thresh_choice)
print("FN at that t =", false_negatives[int(np.argmax(np.array(false_negatives) >= 12))])


## 6. ROC / AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob[:, 1])
plt.figure(figsize=(6.2, 5.6))
plt.plot(fpr, tpr, color="darkorange", label="ROC curve")
idx = list(range(len(thresholds)))[1::4]
for i in idx:
    plt.text(fpr[i], tpr[i], f"{thresholds[i]:.2f}", fontsize=7)
clf = DummyClassifier(strategy="most_frequent", random_state=0)
clf.fit(x_train, y_train)
fpr_d, tpr_d, _ = roc_curve(y_test, clf.predict_proba(x_test)[:, 1])
auc_d = roc_auc_score(y_test, clf.predict_proba(x_test)[:, 1])
plt.plot(fpr_d, tpr_d, color="navy", ls="--", label=f"Dummy most-frequent (AUC={auc_d:.2f})")
plt.xlabel("False Positive Rate (loyal guest flagged)")
plt.ylabel("True Positive Rate (true cancel caught)")
plt.title("ROC — 5-feature cancellation logit")
plt.grid(True, alpha=0.4); plt.legend(loc="lower right")
plt.show()
roc_auc = roc_auc_score(y_test, y_pred_prob[:, 1])
print("ROC AUC:", roc_auc)  # ≈ 0.678


## 7. Class imbalance

In [ ]:
x_train_u, x_test_u, y_train_u, y_test_u = train_test_split(
    df[core], df[outcome], random_state=6, test_size=0.3
)
print("unstrat train pos", float(y_train_u.mean()), "test pos", float(y_test_u.mean()))
x_train_str, x_test_str, y_train_str, y_test_str = train_test_split(
    df[core], df[outcome], random_state=6, test_size=0.3, stratify=df[outcome]
)
print("strat   train pos", float(y_train_str.mean()), "test pos", float(y_test_str.mean()))
str_train_positivity_rate = float(y_train_str.mean())
str_test_positivity_rate = float(y_test_str.mean())
print(str_train_positivity_rate, str_test_positivity_rate)


In [ ]:
log_reg.fit(x_train_str, y_train_str)
print("stratified recall, acc:",
      recall_score(y_test_str, log_reg.predict(x_test_str)),
      accuracy_score(y_test_str, log_reg.predict(x_test_str)))
log_reg.fit(x_train_u, y_train_u)
print("unstrat (rs=6) recall, acc:",
      recall_score(y_test_u, log_reg.predict(x_test_u)),
      accuracy_score(y_test_u, log_reg.predict(x_test_u)))
log_reg_bal = LogisticRegression(
    penalty=None, fit_intercept=True, max_iter=4000, solver="lbfgs",
    class_weight="balanced",
)
log_reg_bal.fit(x_train_u, y_train_u)
print("balanced recall, acc:",
      recall_score(y_test_u, log_reg_bal.predict(x_test_u)),
      accuracy_score(y_test_u, log_reg_bal.predict(x_test_u)))
# Balanced weights are the cheap lever when t=0.5 recall is stuck near 0.37.


## 8. Alternate code

In [ ]:
try:
    import statsmodels.api as sm
    print(sm.Logit(y_train, sm.add_constant(x_train)).fit(disp=False).summary())
except Exception as exc:
    print("statsmodels unavailable or failed:", exc)
    print("sklearn coef", coefficients, "intercept", intercept)


In [ ]:
pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000),
)
pipe.fit(x_train, y_train)
print("scaled-pipeline acc", accuracy_score(y_test, pipe.predict(x_test)))
print("scaled-pipeline rec", recall_score(y_test, pipe.predict(x_test)))


In [ ]:
def predict_at(proba, t=0.5):
    return (np.asarray(proba) >= t).astype(int)
for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    pred = predict_at(y_pred_prob[:, 1], t)
    cm = confusion_matrix(y_test, pred)
    print(f"t={t:.1f}  FN={cm[1,0]:2d}  FP={cm[0,1]:2d}  rec={recall_score(y_test, pred):.3f}")


## 9. More practice

In [ ]:
extra = core + ["is_weekend", "nights"]
Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(df[extra], df[outcome], random_state=0, test_size=0.3)
m_e = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000)
m_e.fit(Xtr_e, ytr_e)
pe = m_e.predict_proba(Xte_e)[:, 1]
print("extra recall", recall_score(yte_e, m_e.predict(Xte_e)))
print("extra AUC   ", roc_auc_score(yte_e, pe))
print("core  AUC   ", roc_auc)


In [ ]:
sessions = pd.read_csv("data/tour_sessions.csv")
print(sessions.head())
print("positivity", sessions.booked.mean())
print("max features (10-EPV)", sessions.booked.value_counts().min() / 10)
print("session vs scroll r =", sessions[["session_min", "scroll_depth"]].corr().iloc[0, 1])
Sx = sessions[["page_views", "session_min", "promo", "repeat_visitor", "mobile"]]
Sy = sessions["booked"]
Str, Ste, sytr, syte = train_test_split(Sx, Sy, random_state=0, test_size=0.3, stratify=Sy)
m_s = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000)
m_s.fit(Str, sytr)
print("coef", m_s.coef_, "intercept", m_s.intercept_)
sp = m_s.predict_proba(Ste)[:, 1]
print("session recall", recall_score(syte, m_s.predict(Ste)), "auc", roc_auc_score(syte, sp))
for t in np.linspace(0.15, 0.7, 12):
    cm = confusion_matrix(syte, (sp >= t).astype(int))
    print(f"t={t:.2f} FN={cm[1,0]:2d} FP={cm[0,1]:2d}")


In [ ]:
weekend = (
    "95% occupancy weekend: an empty room is very expensive. Drive the threshold down "
    "(or use balanced weights) and quote cancel-catch-rate plus courtesy-hold volume."
)
shoulder = (
    "Shoulder midweek: overbooking risk is lower. A higher threshold is fine — "
    "you would rather not annoy a guest who was going to show."
)
board = (
    "Tourism-board pack: one AUC, one catch-rate at the agreed SOP cut, and the "
    "hold-volume that cut implies. No coefficient dump."
)
print(weekend); print(shoulder); print(board)


## 10. Simulation

In [ ]:
# --- editable parameters ---
N = 740
N_REPS = 20
NOISE = 0.00
T = 0.50
TEST_SIZE = 0.30
SEED = 0
# ---------------------------
rng = np.random.default_rng(SEED)
rows_def, rows_bal = [], []
pool_X = df[core].to_numpy(); pool_y = df[outcome].to_numpy()
for r in range(N_REPS):
    idx = rng.choice(len(df), size=N, replace=(N > len(df)))
    X, y = pool_X[idx], pool_y[idx]
    try:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r, stratify=y)
    except ValueError:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r)
    if NOISE > 0:
        flip = rng.random(len(ytr)) < NOISE
        ytr = ytr.copy(); ytr[flip] = 1 - ytr[flip]
    for tag, cw in (("default", None), ("balanced", "balanced")):
        m = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000, class_weight=cw)
        m.fit(Xtr, ytr)
        proba = m.predict_proba(Xte)[:, 1]
        pred = (proba >= T).astype(int)
        rec = dict(
            recall=recall_score(yte, pred, zero_division=0),
            accuracy=accuracy_score(yte, pred),
            auc=roc_auc_score(yte, proba) if len(np.unique(yte)) == 2 else np.nan,
        )
        (rows_def if tag == "default" else rows_bal).append(rec)
sim_def = pd.DataFrame(rows_def); sim_bal = pd.DataFrame(rows_bal)
print("DEFAULT\n", sim_def.agg(["mean", "std"]).round(3))
print("BALANCED\n", sim_bal.agg(["mean", "std"]).round(3))
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, col in zip(axes, ["recall", "accuracy", "auc"]):
    ax.boxplot([sim_def[col].dropna(), sim_bal[col].dropna()], labels=["default", "balanced"])
    ax.set_title(col); ax.set_ylim(0.30, 1.02)
fig.suptitle(f"N={N}  T={T}  noise={NOISE}  reps={N_REPS}", y=1.03)
plt.tight_layout(); plt.show()


## 11. Audience rewrite

In [ ]:
expert = (
    "On the rs=0 hold-out (n=222) an unpenalized 5-feature logit attains AUC 0.68 — "
    "typical for a short hospitality cancel card. Operating at t=0.25 rather than 0.50 "
    "moves the CM from (FN=51, FP=27) to (FN=8, FP=102). booking_value was dropped "
    "(≈ nights × ADR). checkin_hour fails the logit-linearity screen. "
    "class_weight='balanced' lifts rs=6 recall from ~0.37 to ~0.68."
)
technician = (
    "Encode CANCEL/STAY as 1/0. Fit LogisticRegression(penalty=None) on lead_time, "
    "prev_cancels, online_channel, has_deposit, special_requests. Score with "
    "predict_proba[:,1]. If the SOP is 'miss at most ~12 cancels on this test list', "
    "sit near t=0.25. Re-check booking_id uniqueness after the special_requests filter."
)
executive = (
    "A five-signal booking score is useful but not magic (AUC 0.68). The default 50% cut "
    "still misses most cancels on the test list. Dropping the cut to 25% catches almost "
    "all of them and puts many true stayers on a courtesy-hold list. That hold-volume "
    "versus empty-room trade is a GM / revenue-manager choice, not a software default."
)
nonspecialist = (
    "Some reservations are more likely to disappear — booked far ahead, booked online, "
    "the guest has cancelled before, no deposit, no special requests. The model turns "
    "those clues into a 0–100 risk score. Using a cautious cut-off, the hotel keeps a "
    "few extra rooms in reserve so fewer beds sit empty when plans change."
)
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)


## 12. Applications

In [ ]:
applications = [
    "1. Hotel / OTA booking-cancellation scoring (this notebook).",
    "2. Tour-package website conversion (the practice file).",
    "3. Attraction or restaurant no-show prediction from party size + lead time + channel.",
    "4. Repeat-visit / loyalty-enrolment propensity after a stay.",
    "5. Ancillary upsell (spa, transfer, breakfast) accept vs decline.",
    "6. Flight-plus-hotel package abandon at the payment step.",
    "7. Destination-recommendation click / not-click on a DMO site.",
    "8. Review-invite response (will this guest leave a review?).",
    "9. Group-booking confirm vs release as the cut-off date approaches.",
    "10. Weather-sensitive outdoor activity go / no-go attendance.",
]
not_appropriate = [
    "1. Choosing among 12 destinations (multiclass / ranking, not binary logit).",
    "2. Repeated stays by the same loyalty member without a clustered model.",
    "3. A boutique inn with 40 labelled cancels and 15 features — EPV collapse.",
    "4. Raw booking-engine clickstreams as columns — learn a representation first.",
    "5. Causal claim that 'dropping the deposit caused cancels' — this is a risk score.",
    "6. Daily occupancy *level* forecasting — that is a time series, not a 0/1 logit.",
    "7. Highly seasonal demand with no calendar features and no refit.",
    "8. Length-of-stay or spend as the real target — use a regression / duration model.",
]
for row in applications: print(row)
print("--- not appropriate ---")
for row in not_appropriate: print(row)


## 13. Saved figures

`logreg_tour_flowchart.png`, `logreg_tour_boxplot.png`, `logreg_tour_boxplot_filtered.png`,
`logreg_tour_logit_curves.png`, `logreg_tour_corr_heatmap.png`, `logreg_tour_proba_hist.png`,
`logreg_tour_roc.png`, `logreg_tour_threshold_sweep.png`, `logreg_tour_simulation.png`.
